# CDI-Adapter on Google Colab (GPU)

Builds the whole stack on **your own** Colab machine: Postgres 16, Redis 7, MinIO, the Qwen2.5-VL model
on the GPU, and the CareFlow website.

**Before you start**
- Runtime → Change runtime type → **L4 GPU** (A100 also works).
- **Synthetic / test data only.** Colab is hosted outside India, so no real patient data (PHI) ever goes here.
- Keep this tab open. Colab wipes the machine after ~90 minutes idle; then run all cells again.
- Everything below runs on *your* Colab machine only. Services listen on `127.0.0.1`, so nothing
  outside this machine can connect to them, and nobody else can reach your copy.
- **Never share this notebook with outputs.** Step 8 prints passwords. Before sharing:
  Edit → Clear all outputs.

Run the cells top to bottom (about 15 minutes the first time).

## Step 1 - Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!cat /etc/os-release | head -2; /usr/bin/python3.12 -V

## Step 2 - Get the source code

Either:
- **A (recommended):** add a Colab secret (key icon on the left) named `GITHUB_TOKEN` with read access to
  `manishtech0607-cmyk/CDI-Adapter`, and turn on notebook access. Survives machine wipes.
- **B:** upload `cdi-src.tar.gz` to `/content` with the Files panel (folder icon on the left).

In [ ]:
import os, subprocess
REPO = 'github.com/manishtech0607-cmyk/CDI-Adapter.git'
BRANCH = 'dev_chakravardhan'
if os.path.isdir('/content/cdi/src'):
    print('source already present')
else:
    try:
        from google.colab import userdata
        tok = userdata.get('GITHUB_TOKEN')
    except Exception:
        tok = None
    if tok:
        r = subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH,
                            f'https://x-access-token:{tok}@{REPO}', '/content/cdi'],
                           capture_output=True, text=True)
        print('clone exit', r.returncode, r.stderr[-400:].replace(tok, '***'))
        # do not leave the token in .git/config
        subprocess.run(['git', '-C', '/content/cdi', 'remote', 'set-url', 'origin', f'https://{REPO}'])
    elif os.path.exists('/content/cdi-src.tar.gz'):
        os.makedirs('/content/cdi', exist_ok=True)
        subprocess.run(['tar', '-xzf', '/content/cdi-src.tar.gz', '-C', '/content/cdi'], check=True)
        print('extracted cdi-src.tar.gz')
    else:
        print('NO SOURCE: add the GITHUB_TOKEN secret or upload cdi-src.tar.gz to /content, then rerun')
if os.path.isdir('/content/cdi'):
    print(sorted(os.listdir('/content/cdi')))

## Step 3 - Build the services (Postgres, Redis, MinIO) - about 5 minutes

Expected at the end: `minio OK`, `pg OK`. Lines starting `DRIFT: fhir package changed` are a known,
harmless issue (backlog E0-S16).

In [ ]:
!bash /content/cdi/infra/colab/colab_setup.sh > /content/setup.log 2>&1; grep -e '^===' -e ' OK' -e 'DOWN' -e 'fhir packages' /content/setup.log

## Step 4 - Install GPU PyTorch and switch the model to the GPU - a few minutes

In [ ]:
%%bash
set -e
cd /content/cdi
. infra/runpod/versions.env
.venv/bin/python -m pip install -q torch==$TORCH_VERSION torchvision==$TORCHVISION_VERSION --index-url https://download.pytorch.org/whl/cu128
.venv/bin/python -c 'import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())'
sed -i 's/^CDI_MLSERVE_BACKEND=.*/CDI_MLSERVE_BACKEND=hf/' .env
grep BACKEND .env

## Step 5 - Start the model server and the website

Expected: `mlserve up`, then `app tier up: balancer :8888 -> 127.0.0.1:8881 127.0.0.1:8882` and a
`"status":"ok"` line. The website runs as two copies behind a balancer, with two job workers
(backlog E10-S9), so one copy can fail without anyone noticing. The model itself (~16 GB) downloads
and loads on the first document, so the first upload takes several minutes.

In [ ]:
import os, subprocess, time
from google.colab.output import eval_js
# Colab's proxy rewrites Host, so the website must be told its public hostname or it refuses
# every form post, sign-in included (CDI_PUBLIC_ORIGINS, backlog E11-S1)
host = eval_js('google.colab.kernel.proxyPort(8888)').split('//', 1)[1].rstrip('/')
lines = [l for l in open('/content/cdi/.env').read().splitlines() if not l.startswith('CDI_PUBLIC_ORIGINS=')]
open('/content/cdi/.env', 'w').write('\n'.join(lines + ['CDI_PUBLIC_ORIGINS=' + host]) + '\n')
print('public origin set')
env = dict(os.environ, WS='/workspace', REPO='/content/cdi', HF_HOME='/workspace/hf-cache', PATH='/opt/cdi-shim:' + os.environ['PATH'])
subprocess.run(['pkill', '-f', 'cdi_adapter.mlserve']); time.sleep(2)
# Output goes to a file, never a pipe: older copies of start_mlserve.sh held a pipe open for the server's lifetime.
subprocess.run('bash infra/runpod/start_mlserve.sh > /workspace/logs/start_mlserve.out 2>&1', shell=True, cwd='/content/cdi', env=env, stdin=subprocess.DEVNULL)
print(open('/workspace/logs/start_mlserve.out').read())
# E10-S9: two website copies + two job workers behind the balancer on :8888 (start_apptier.sh)
subprocess.run('bash infra/runpod/start_apptier.sh > /workspace/logs/start_apptier.out 2>&1', shell=True, cwd='/content/cdi', env=env, stdin=subprocess.DEVNULL)
print(open('/workspace/logs/start_apptier.out').read()[-600:])
print(subprocess.run('curl -s -m 10 http://127.0.0.1:8077/healthz; echo; curl -s -m 10 http://127.0.0.1:8888/healthz', shell=True, capture_output=True, text=True).stdout)

## Step 6 - Create your sign-in account

Run it, then type a password in the box. It is never printed or saved in the notebook.

In [ ]:
import getpass, subprocess
pw = getpass.getpass('Choose a password for user  admin : ')
p = subprocess.run(['/content/cdi/.venv/bin/python', '-m', 'cdi_adapter.webapp.users', 'add', 'admin',
                    '--name', 'Clinic Admin', '--role', 'admin'],
                   input=pw + chr(10), capture_output=True, text=True, cwd='/content/cdi')
print(p.stdout.strip() or p.stderr.strip())
print('exit', p.returncode)

## Step 7 - Website URL

Open it in the same browser where you are signed in to Colab (only your Google account can open it).
Sign in as `admin`.

In [ ]:
from google.colab.output import eval_js
print(eval_js('google.colab.kernel.proxyPort(8888)'))

## Step 8 - Connection details for THIS machine (MinIO, Redis, GPU)

**Contains passwords. Clear this output before sharing the notebook.** They are regenerated every time
the machine is rebuilt. They only work from inside this Colab machine (`127.0.0.1`).

In [ ]:
%%bash
cd /content/cdi
g() { .venv/bin/python src/cdi_adapter/secretstore.py --dir /run/cdi-secrets get "$@"; }
echo "== MinIO =="
echo "health        : $(curl -s -o /dev/null -w '%{http_code}' http://127.0.0.1:9000/minio/health/live) (200 = OK)"
grep -E '^CDI_S3_(ENDPOINT_URL|BUCKET)=' .env
echo "CDI_S3_ACCESS_KEY=$(g cdi_s3_access_key)"
echo "CDI_S3_SECRET_KEY=$(g cdi_s3_secret_key)"
echo "console       : http://127.0.0.1:9001"
echo
echo "== Redis =="
echo "CDI_REDIS_URL=$(g cdi_redis_url)"
echo "ping          : $(REDISCLI_AUTH=$(g cdi_redis_url --part password) redis-cli -h 127.0.0.1 -p 6379 ping)"
echo "(redis-cli -u <url> fails with WRONGPASS on the empty username; use REDISCLI_AUTH + -h/-p)"
echo
echo "== GPU =="
nvidia-smi --query-gpu=name,memory.total,memory.used,driver_version,utilization.gpu --format=csv
.venv/bin/python -c 'import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())'
echo "model server: $(curl -s http://127.0.0.1:8077/healthz)"

## Step 9 - Test URL anyone can open (share it with testers)

Opens a Cloudflare quick tunnel to the website and prints a `https://….trycloudflare.com` address. It works
in any browser, without this Colab window or a Google sign-in; testers sign in with a website account.
The address is new every time the tunnel starts. `status` / `stop` are the other commands.

**Test data only (synthetic or de-identified):** the traffic is decrypted at Cloudflare.

In [ ]:
!bash /content/cdi/infra/colab/start_test_tunnel.sh

## Step 10 - Update this machine to new code

The code here must be exactly the verified local code, never hand-edited. After the local checks pass,
the developer builds `cdi-src.tar.gz` (`python scripts/dev/colab_bundle.py`). Upload it with the Files
panel on the left into `/content`, then run this. It checks every file, applies the tree, migrates the
database and restarts the website. Expected: `SYNCED: bundle <id>`.

In [ ]:
!tar -xzf /content/cdi-src.tar.gz -C /tmp infra/colab/sync.sh && bash /tmp/infra/colab/sync.sh /content/cdi-src.tar.gz

## If something goes wrong
- **URL stopped working / files gone:** the machine was wiped after being idle. Run all cells again.
- **Step 3 says `DOWN`:** read `/workspace/logs/bootstrap.log`.
- **Website errors:** `!bash /content/cdi/infra/runpod/start_apptier.sh status; tail -40 /workspace/logs/webapp-1.log`; model: `!tail -40 /workspace/logs/mlserve.log`.
- **Documents stay `draft`:** Colab runs the official HAPI FHIR validator, which currently flags two
  conformance issues in our bundles (known, being fixed).